# Day 14 — Design Pattern: **Strategy**

**Phase 1 · Module 2: AWS Bedrock & AgentCore** · ~30 min

Tomorrow-ish (Day 17) you'll rip Azure OpenAI out of a LangGraph agent and drop **Bedrock** in. If the call site is a pile of `if provider == "bedrock": ... elif provider == "azure": ...`, that swap touches every function that talks to a model. The **Strategy pattern** fixes this: define **one interface** for "generate text", give each provider its **own implementation**, and let the calling code depend only on the interface. Swapping providers becomes a one-line change — no edits to business logic.

> **Runnable & keyless.** The provider classes return canned strings instead of real API calls, so every cell runs offline. Each `generate()` is exactly where a real `boto3` / `openai` call would go — marked in comments.

## 1. The smell — branching on the provider everywhere

Here's the anti-pattern: the *decision* about which SDK to call is baked into the business function. Add a third provider (Google) and you edit this function — and every other function shaped like it. That's the Open/Closed principle violated: you can't extend without modifying.

In [1]:
# ❌ anti-pattern — provider logic tangled into the caller
def summarise_bad(text, provider):
    if provider == "bedrock":
        # boto3 bedrock-runtime.converse(...) would go here
        out = f"[bedrock/claude] summary of {len(text)} chars"
    elif provider == "azure":
        # openai.AzureOpenAI().chat.completions.create(...) here
        out = f"[azure/gpt-4o] summary of {len(text)} chars"
    else:
        raise ValueError(f"unknown provider {provider!r}")
    return out

print(summarise_bad("...banking policy doc...", "bedrock"))
print(summarise_bad("...banking policy doc...", "azure"))

[bedrock/claude] summary of 24 chars
[azure/gpt-4o] summary of 24 chars


☝ Every business function that calls a model needs this same `if/elif` block, and adding a provider means editing all of them. The provider choice has leaked into code that should only care about *summarising*. Strategy pulls that choice out.

## 2. The interface + interchangeable implementations

**Strategy** = one abstract "how" and many concrete versions of it. Define an `LLMProvider` interface with a single `generate(prompt) -> str` method, then implement it once per provider. Each class hides its own SDK details behind the shared method name.

In [2]:
from abc import ABC, abstractmethod

class LLMProvider(ABC):                     # the Strategy interface
    @abstractmethod
    def generate(self, prompt: str) -> str: ...

class BedrockProvider(LLMProvider):
    model = "eu.anthropic.claude-sonnet-4-5"
    def generate(self, prompt: str) -> str:
        # real: boto3.client("bedrock-runtime").converse(modelId=self.model, ...)
        return f"[bedrock:{self.model.split('.')[-1]}] → {prompt[:40]!r}"

class AzureProvider(LLMProvider):
    model = "gpt-4o"
    def generate(self, prompt: str) -> str:
        # real: AzureOpenAI().chat.completions.create(model=self.model, ...)
        return f"[azure:{self.model}] → {prompt[:40]!r}"

for p in (BedrockProvider(), AzureProvider()):
    print(p.generate("Summarise the loan policy"))

[bedrock:claude-sonnet-4-5] → 'Summarise the loan policy'
[azure:gpt-4o] → 'Summarise the loan policy'


☝ Two classes, **same method signature**. The `ABC` + `@abstractmethod` means Python refuses to instantiate a provider that forgot to implement `generate` — the interface is enforced. Each `generate` is the only place its SDK appears.

## 3. The context depends on the **interface**, not the concrete class

The **context** is the code that *uses* a strategy. It's handed an `LLMProvider` and calls `.generate()` — it neither knows nor cares which provider it got. This is the whole payoff: business logic written once, works with any provider present or future.

In [3]:
class Summariser:                      # the context
    def __init__(self, provider: LLMProvider):
        self.provider = provider           # depends on the interface only

    def run(self, doc: str) -> str:
        prompt = f"Summarise for a Barclays analyst:\n{doc}"
        return self.provider.generate(prompt)

doc = "Customer overdraft policy v3 ..."
print(Summariser(BedrockProvider()).run(doc))
print(Summariser(AzureProvider()).run(doc))   # same Summariser code, different engine

[bedrock:claude-sonnet-4-5] → 'Summarise for a Barclays analyst:\nCustom'
[azure:gpt-4o] → 'Summarise for a Barclays analyst:\nCustom'


☝ `Summariser` has **zero** provider branching — compare it to §1. The Day-17 migration (Azure → Bedrock) is now: construct `Summariser` with a different strategy. The business method never changes. That's dependency **inversion**: high-level code depends on the abstraction, not the SDK.

## 4. Config-driven selection — pick the strategy at runtime

You rarely `new` a provider by hand; you choose it from **config** (an env var, a settings object from Day 13). A small registry maps a string to a strategy, so ops can flip `LLM_PROVIDER=bedrock` → `azure` with **no code change and no redeploy of logic**.

In [4]:
import os

REGISTRY = {                               # name -> strategy class
    "bedrock": BedrockProvider,
    "azure":   AzureProvider,
}

def make_provider(name: str) -> LLMProvider:
    try:
        return REGISTRY[name]()            # look up + construct
    except KeyError:
        raise ValueError(f"unknown provider {name!r}; have {list(REGISTRY)}")

os.environ["LLM_PROVIDER"] = "bedrock"     # pretend this came from the environment
active = make_provider(os.environ["LLM_PROVIDER"])
print("active provider:", type(active).__name__)
print(Summariser(active).run(doc))

# a new provider = add ONE line to REGISTRY, touch no business code
class GoogleProvider(LLMProvider):
    def generate(self, prompt): return "[google:gemini] → " + repr(prompt[:40])
REGISTRY["google"] = GoogleProvider
print(Summariser(make_provider("google")).run(doc))

active provider: BedrockProvider
[bedrock:claude-sonnet-4-5] → 'Summarise for a Barclays analyst:\nCustom'
[google:gemini] → 'Summarise for a Barclays analyst:\nCustom'


☝ Adding **Google** was one registry entry — `Summariser`, `make_provider`, and every caller stayed untouched. That's *open for extension, closed for modification*. Switching the whole app's provider is now an env-var flip.

## 5. The Pythonic shortcut — `Protocol` (duck typing)

Python doesn't *need* the `ABC` base class: if an object has a `generate(prompt) -> str` method, it already works with `Summariser`. `typing.Protocol` gives you the **type-checker's** guarantee (structural typing) without forcing providers to inherit anything — useful for wrapping a third-party client you can't subclass.

In [5]:
from typing import Protocol

class SupportsGenerate(Protocol):
    def generate(self, prompt: str) -> str: ...

class ThirdPartyClient:                    # inherits nothing, yet fits the Protocol
    def generate(self, prompt: str) -> str:
        return "[vendor-sdk] " + prompt[:30]

def summarise(doc: str, llm: SupportsGenerate) -> str:   # accepts anything shaped right
    return llm.generate("Summarise: " + doc)

print(summarise(doc, ThirdPartyClient()))
print(summarise(doc, BedrockProvider()))   # the ABC version fits too

[vendor-sdk] Summarise: Customer overdraft 
[bedrock:claude-sonnet-4-5] → 'Summarise: Customer overdraft policy v3 '


☝ No inheritance, still type-safe — `ThirdPartyClient` satisfies `SupportsGenerate` just by having the right method. Use **`ABC`** when you own the providers and want to *enforce* the contract at construction; use **`Protocol`** when you're adapting objects you don't control. Both are Strategy.

## 6. Recap + checklist

| Piece | Role in Strategy |
|---|---|
| `LLMProvider` (ABC / Protocol) | the **interface** — one method, `generate` |
| `BedrockProvider`, `AzureProvider` | interchangeable **strategies** |
| `Summariser` | the **context** — depends on the interface only |
| `REGISTRY` + `make_provider` | **config-driven** strategy selection |
| adding `GoogleProvider` | extend by adding, not editing (Open/Closed) |

**Your 30 min today**
- [ ] Write an `LLMProvider` ABC with a `generate()` method.
- [ ] Implement two strategies (Bedrock-shaped, Azure-shaped) returning canned strings.
- [ ] Write a context class that takes a provider and never branches on its type.
- [ ] Select the strategy from an env var via a registry; add a third provider with one line.
